# 04 — Model Benchmarking & Stratified 5-Fold Cross-Validation

**Comparative study across 7 ML paradigms**

We evaluate 7 classifiers using stratified 5-fold CV to get reliable, bias-corrected performance estimates. The **primary metric is Macro F0.5**, which weights precision twice as much as recall — appropriate for entity resolution where false positives (merging different businesses) corrupt the downstream database.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import Image, display

sns.set_theme(style='whitegrid')
%matplotlib inline
print("Setup complete.")

### 1. Model Leaderboard

Sorted by **Macro F0.5** (descending). Mean ± std across 5 folds. Lower std = more stable across different data splits.

In [ ]:
metrics = pd.read_csv('results/tables/model_comparison_metrics.csv')
metrics_sorted = metrics.sort_values('Macro_F0.5_Mean', ascending=False).reset_index(drop=True)
metrics_sorted.index += 1

# Display with color gradient on key metrics
cols_to_color = ['Macro_F0.5_Mean', 'AUC_ROC_Mean', 'Precision_Mean', 'Recall_Mean']
cols_to_color = [c for c in cols_to_color if c in metrics_sorted.columns]

metrics_sorted.style\
    .background_gradient(cmap='YlGn', subset=cols_to_color)\
    .format({c: '{:.4f}' for c in metrics_sorted.select_dtypes('number').columns})

### 2. Why F0.5? The Metric Choice

The F-beta score is defined as:

$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

With **β = 0.5**, precision is weighted **twice as heavily** as recall.

**Why does this matter for entity resolution?**

- A **false positive** (predicting two *different* businesses as a match) merges distinct records → corrupts the entity database, causes incorrect deduplication, and is very hard to undo.
- A **false negative** (missing a true duplicate) leaves a redundant record → less harmful; can be caught in later audits.

Therefore, we penalize false positives more heavily by choosing F0.5 as our primary metric.

### 3. Leaderboard Bar Chart

Horizontal bar chart sorted by Macro F0.5 Mean. Error bars show ±1 standard deviation across folds.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models  = metrics_sorted['Model'].str.replace('_', ' ')
means   = metrics_sorted['Macro_F0.5_Mean']
stds    = metrics_sorted['Macro_F0.5_Std'] if 'Macro_F0.5_Std' in metrics_sorted.columns else [0]*len(models)

colors = sns.color_palette('RdYlGn', len(models))
bars = ax.barh(models[::-1], means[::-1], xerr=stds[::-1],
               color=colors, edgecolor='black', linewidth=0.6,
               capsize=4, height=0.6)

for bar, val in zip(bars, means[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Macro F0.5 Mean (5-Fold CV)', fontsize=12)
ax.set_title('Model Leaderboard — Macro F0.5 Score', fontsize=14, fontweight='bold')
ax.set_xlim(0, min(1.0, means.max() + 0.08))
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/model_leaderboard_f05.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. ROC & Precision-Recall Curves

These curves were generated during 5-fold CV and saved to `results/plots/`.

In [ ]:
for fname in ['roc_curves_comparison.png', 'pr_curves_comparison.png']:
    fpath = f'results/plots/{fname}'
    if os.path.exists(fpath):
        display(Image(fpath))
    else:
        print(f'[INFO] {fpath} not found — run model training pipeline to generate it.')

### 5. Confusion Matrices

Confusion matrices show the per-class breakdown of model predictions. We expect hard-negative pairs to be the main source of errors.

In [ ]:
fpath = 'results/plots/confusion_matrices_grid.png'
if os.path.exists(fpath):
    display(Image(fpath))
else:
    print(f'[INFO] {fpath} not found — run model training pipeline to generate it.')

### 6. Multi-Metric Radar Summary

The radar (spider) chart gives a holistic view of each model's strengths and weaknesses across five metrics simultaneously: Precision, Recall, Macro F1, Macro F0.5, and AUC-ROC.

In [ ]:
metric_cols = ['Precision_Mean', 'Recall_Mean', 'Macro_F1_Mean', 'Macro_F0.5_Mean', 'AUC_ROC_Mean']
metric_cols = [c for c in metric_cols if c in metrics.columns]
labels = [c.replace('_Mean', '').replace('_', ' ') for c in metric_cols]

N = len(metric_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
palette = sns.color_palette('tab10', len(metrics))

for i, (_, row) in enumerate(metrics.iterrows()):
    values = [row[c] for c in metric_cols]
    values += values[:1]
    model_label = row['Model'].replace('_', ' ')
    ax.plot(angles, values, linewidth=2, linestyle='solid',
            label=model_label, color=palette[i])
    ax.fill(angles, values, alpha=0.07, color=palette[i])

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('Multi-Metric Radar: All Models', fontsize=14,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig('results/plots/radar_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()